### Notebook for trying different things and learning

In [2]:
test_string = "hello! こんにちは!"

In [7]:
utf8_encoded = test_string.encode("utf-8")
print(utf8_encoded)
bytes_list = list(utf8_encoded)
print("In characters ", len(test_string), " number of characters")
print(bytes_list)
print("In UTF8 ", len(bytes_list), " characters")
bytes_list = list(test_string.encode("utf-16"))
print(bytes_list)
print("In UTF16 ", len(bytes_list), " characters")
bytes_list = list(test_string.encode("utf-32"))
print(bytes_list)
print("In UTF32 ", len(bytes_list), " characters")

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'
In characters  13  number of characters
[104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227, 129, 171, 227, 129, 161, 227, 129, 175, 33]
In UTF8  23  characters
[255, 254, 104, 0, 101, 0, 108, 0, 108, 0, 111, 0, 33, 0, 32, 0, 83, 48, 147, 48, 107, 48, 97, 48, 111, 48, 33, 0]
In UTF16  28  characters
[255, 254, 0, 0, 104, 0, 0, 0, 101, 0, 0, 0, 108, 0, 0, 0, 108, 0, 0, 0, 111, 0, 0, 0, 33, 0, 0, 0, 32, 0, 0, 0, 83, 48, 0, 0, 147, 48, 0, 0, 107, 48, 0, 0, 97, 48, 0, 0, 111, 48, 0, 0, 33, 0, 0, 0]
In UTF32  56  characters


In [9]:
print(ord('h'))

104


In [4]:
import regex as re
pat_oai =  r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

text = "Once upon a time, in a warm and sunny place, there was a big pit. A little boy named Tom liked to play near the pit. One day, Tom lost his red ball. He was very sad."
compiled = re.compile(pat_oai)
matches = compiled.finditer(text)
for match in matches:
    print(match.group())


Once
 upon
 a
 time
,
 in
 a
 warm
 and
 sunny
 place
,
 there
 was
 a
 big
 pit
.
 A
 little
 boy
 named
 Tom
 liked
 to
 play
 near
 the
 pit
.
 One
 day
,
 Tom
 lost
 his
 red
 ball
.
 He
 was
 very
 sad
.


In [55]:
# BPE Implementation
from typing import BinaryIO
import os
import multiprocessing as mp
import regex as re

def find_chunk_boundaries(
        file: BinaryIO,
        desired_num_chunks: int,
        split_special_token: bytes,
    ) -> list[int]:
        """
        Chunk the file into parts that can be counted independently.
        May return fewer chunks if the boundaries end up overlapping.
        split_special_token is used to create "clean" boundaries that will not have interdependence.
        """
        assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"
        print("split special token for chunking is ", split_special_token)

        # Get total file size in bytes
        file.seek(0, os.SEEK_END)
        file_size = file.tell()
        file.seek(0)

        chunk_size = file_size // desired_num_chunks

        # Initial guesses for chunk boundary locations, uniformly spaced
        # Chunks start on previous index, don't include last index
        chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
        chunk_boundaries[-1] = file_size

        mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

        found_chunk_seps = 0
        for bi in range(1, len(chunk_boundaries) - 1):
            initial_position = chunk_boundaries[bi]
            file.seek(initial_position)  # Start at boundary guess
            while True:
                mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

                # If EOF, this boundary should be at the end of the file
                if mini_chunk == b"":
                    chunk_boundaries[bi] = file_size
                    break

                # Find the special token in the mini chunk
                found_at = mini_chunk.find(split_special_token)
                if found_at != -1:
                    chunk_boundaries[bi] = initial_position + found_at
                    found_chunk_seps += 1
                    break
                initial_position += mini_chunk_size

        # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
        return sorted(set(chunk_boundaries))

def _process_chunk(args: tuple[str, int, int, list[str]]) -> list[str]:
    filepath, start, end, special_tokens_str = args
    with open(filepath, "rb") as f:
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")

    pat_oai = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    compiled = re.compile(pat_oai)

    # Split out special tokens before running regex pre-tokenization
    if special_tokens_str:
        pattern = "|".join(re.escape(tok) for tok in special_tokens_str)
        segments = re.split(pattern, chunk)
    else:
        segments = [chunk]

    chunk_tokens = []
    for segment in segments:
        if segment:
            chunk_tokens.extend(compiled.findall(segment))
    return chunk_tokens

def pre_tokenize(
    filepath: str,
    special_tokens: list[bytes],
    doc_separator: bytes,
    num_processes: int = 4,
) -> list[str]:
    with open(filepath, "rb") as f:
        boundaries = find_chunk_boundaries(f, num_processes, doc_separator)
        print("Chunk boundaries: ", boundaries)

    special_tokens_str = [tok.decode("utf-8", errors="ignore") for tok in special_tokens]
    tasks = [
        (filepath, start, end, special_tokens_str)
        for start, end in zip(boundaries[:-1], boundaries[1:])
    ]

    # Use fork context on macOS to avoid spawn __main__ pickling issues in Jupyter
    try:
        ctx = mp.get_context("fork")
    except ValueError:
        ctx = mp.get_context()

    with ctx.Pool(processes=num_processes) as pool:
        chunk_results = pool.map(_process_chunk, tasks)

    pre_tokens = []
    for res in chunk_results:
        pre_tokens.extend(res)
    return pre_tokens

In [56]:
# Observing byte encodings for a special token we create
special_token = bytes("<|endoftext|>", "utf-8")
special_token_encoding = "<|endoftext|>".encode("utf-8")
print(list(special_token))
print(list(special_token_encoding))

[60, 124, 101, 110, 100, 111, 102, 116, 101, 120, 116, 124, 62]
[60, 124, 101, 110, 100, 111, 102, 116, 101, 120, 116, 124, 62]


In [81]:
pre_tokens = pre_tokenize("/Users/dibakarbarua/Desktop/Code/cs336/cs336_basics/data/TinyStoriesV2-GPT4-valid.txt", [special_token], special_token)
print(pre_tokens[0])
print(pre_tokens[1])
print(pre_tokens[2])
print(pre_tokens[3])
print(pre_tokens[-1])
print(pre_tokens[-2])
print(pre_tokens[-3])
print(pre_tokens[-4])

split special token for chunking is  b'<|endoftext|>'
Chunk boundaries:  [0, 5625758, 11252559, 16877372, 22502601]
u
 don
't
 have
.
 happy
 very
 were


In [57]:
%prun pre_tokenize("/Users/dibakarbarua/Desktop/Code/cs336/cs336_basics/data/TinyStoriesV2-GPT4-valid.txt", [special_token], special_token)

split special token for chunking is  b'<|endoftext|>'
Chunk boundaries:  [0, 5625758, 11252559, 16877372, 22502601]
 

         50330 function calls (50200 primitive calls) in 0.884 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
  572/570    0.400    0.001    1.103    0.002 {built-in method posix.read}
        6    0.184    0.031    0.184    0.031 {built-in method _pickle.loads}
       20    0.068    0.003    0.069    0.003 {method 'control' of 'select.kqueue' objects}
      572    0.031    0.000    0.031    0.000 {method 'write' of '_io.BytesIO' objects}
        1    0.025    0.025    0.025    0.025 {function Random.seed at 0x104de2660}
    99/66    0.022    0.000    0.024    0.000 {method 'acquire' of '_thread.lock' objects}
       10    0.015    0.002    0.015    0.002 {method 'extend' of 'list' objects}
        1    0.012    0.012    0.012    0.012 {built-in method _imp.create_dynamic}
        6    0.011    0.002    0.011    0.002 {built-in method marshal.loads}
       20    0.007    0.000    0.077    0.004 selectors.py:558(select)
   

In [74]:
some_str = ' hello'.encode("utf-8")
print(some_str)
some_bytes = [bytes([val]) for val in some_str]
print(some_bytes)
target_token = b'h'
for i, token in enumerate(some_bytes):
    if token == target_token and (idx + 1) != len(some_bytes):
        some_bytes[i] = token + some_bytes[i+1]
        some_bytes.pop(i+1)
print(some_bytes)

b' hello'
[b' ', b'h', b'e', b'l', b'l', b'o']
[b' ', b'he', b'l', b'l', b'o']


In [98]:
def merge(
    filepath: str,
    special_tokens: list[bytes],
    doc_separator: bytes,
):
    ## 1.0 Setup Pre-Tokenization
    special_token = bytes("<|endoftext|>", "utf-8")
    pre_tokens = pre_tokenize(filepath, special_tokens, doc_separator)

    # NOTE: pre_tokens contains a list of all pre-tokens as re.match objects

    ## 2.0 Initialize Vocabulary:
    
    # 2.1 Represent UTF-8 encodings, Integer IDs 0 -> 255
    # NOTE: No character/token representation here, simply integer ID <-> bytes
    utf8_vocab_dict : dict[int, bytes] = {
        byte_val: bytes([byte_val]) for byte_val in range(0, 256)
    }

    # 2.2 Represent special tokens
    # NOTE: Assuming that doc_separator is present in special_tokens
    byte_val_start = 256 # tokens 0-255 allocated to UTF-8
    byte_val_end = byte_val_start + len(special_tokens)
    special_vocab_dict: dict[int, bytes] = {
        byte_val: special_tokens[byte_val - byte_val_start] for byte_val in range(byte_val_start, byte_val_end)
    }

    # 2.3 Any new token captured will be at the index below and incremented upon capture
    byte_val_start = byte_val_end

    ## 3.0 Initialize Primary Merge Containers
    
    # 3.1 Container to store all pre-tokens and their dynamic tokenized bytearrays from the current vocabulary
    # Example: `alone ` as a pre-token will be initially stored as [b'a', b'l', ..... b' ']
    # Example: `alone ` after adding `al` to the vocabulary will be stored as [b'al', b'o', .... b' '] 
    pre_tokens_dict : dict[str, list[bytes]] = {
        token : [bytes([val]) for val in token.encode('utf-8')] for token in pre_tokens
    }

    # 3.2 Container to store all active byte-pairs
    # Example 'al' will be stored as { 'al' : ['alone', 'always', .... ] } for pre-tokens and { 'al' :  4} } for count
    byte_pairs_tokens : dict[bytes, list[str]] = {}
    byte_pairs_counts : dict[bytes, int] = {}

    # 4.0 Main Loop through all pre-tokens
    # 4.1 Form initial byte-pairs for all tokens
    for token in pre_tokens:
        old_bytes = pre_tokens_dict[token]
        new_bytes = []
        only_one_byte = True
        for idx in range(len(old_bytes) - 1):
            if idx == 0:
                only_one_byte = False
            new_byte_pair = old_bytes[idx] + old_bytes[idx + 1]
            new_bytes.append(new_byte_pair)
            # 4.11 Found an existing byte-pair already tracked
            if new_byte_pair in byte_pairs_tokens:
                assert new_byte_pair in byte_pairs_counts
                token_list = byte_pairs_tokens[new_byte_pair]
                if token not in token_list:
                    byte_pairs_tokens[new_byte_pair].append(token)
                byte_pairs_counts[new_byte_pair] += 1
            # 4.12 New byte-pair
            else:
                assert new_byte_pair not in byte_pairs_counts
                byte_pairs_tokens[new_byte_pair] = [token]
                byte_pairs_counts[new_byte_pair] = 0
        if only_one_byte:
            new_bytes = old_bytes
            # only one byte so we dont have to track any byte-pair in this token
    
    # TODO: Test the first merge
    keys_bpe_tokens = list(byte_pairs_tokens.keys())
    keys_bpe_counts = list(byte_pairs_counts.keys())
    assert keys_bpe_counts == keys_bpe_tokens
    print("Keys are same across counts and tokens!")
    for bp in keys_bpe_tokens:
        print("bp : ", str(bp), " tokens: ", byte_pairs_tokens[bp])
    



special_token = "<|endoftext|>".encode("utf-8")
merge("/Users/dibakarbarua/Desktop/Code/cs336/cs336_basics/data/test_data.txt", [special_token], special_token)


split special token for chunking is  b'<|endoftext|>'
Chunk boundaries:  [0, 34783, 69708, 103964, 138420]
Keys are same across counts and tokens!
bp :  b' d'  tokens:  [' don', ' dog', ' day', ' dark', ' deep', ' did', ' decided', ' down', ' during', ' dolls', ' dad', ' decorate', ' draw', ' door', ' drew', ' dresser', ' do', ' dustpan', ' dried', ' dance', ' dreamed', ' days', ' dirty', ' drop', ' different', ' darling', ' dinner', ' disappear', ' dreaming', ' drank', ' done', ' daddy', ' determined', ' doing', ' dry', ' deliver', ' diary', ' daughter', ' dirt', ' didn', ' does', ' dangerous', ' drops', ' disappointed', ' dull', ' discovery', ' dive', ' deeper', ' doctor', ' deaf', ' dream', ' daring', ' danced', ' decide', ' dropped', ' drive', ' drove', ' doll', ' delicious', ' dead', ' dug', ' duck', ' dare', ' dared', ' dust', ' dusty', ' decorating', ' disagreed', ' discovered', ' disagree', ' dress', ' drink', ' delight', ' demand', ' doors', ' dinosaur', ' distant', ' dear', '